**산화머니 | 주택 구매·대출 데이터 전처리 및 검증**

주택 구매 사례에 지역 정보를 추가하고, 구매에 연결된 대출의 원금과 상환액이 올바르게 집계됐는지 확인합니다. 주택 구매, 개별 대출, 지역 정보 병합 노트북의 핵심 작업을 재구성했습니다.

아래 코드는 가상 구매 사례 3건으로 실행됩니다. 실제 프로젝트 데이터에서 확인한 결과는 마지막에 따로 정리했습니다. Python과 pandas가 필요합니다.

**입력 데이터와 결합 기준**

- 주택 구매 자료는 구매 사례 단위이며, 예제에서는 `real_asset_id`로 식별합니다.
- 개별 대출 자료에는 한 구매 사례에 연결된 대출이 여러 건일 수 있습니다. 구매 연결 대출만 자산별로 집계한 뒤 구매 자료와 대조합니다.
- 지역 정보는 고객 ID로 연결합니다. 같은 고객의 지역 정보가 서로 다르면 병합 전에 확인합니다.

In [1]:
import pandas as pd
from pandas.testing import assert_frame_equal

home = pd.DataFrame({
    "customer_id": ["demo-003", "demo-001", "demo-002"],
    "real_asset_id": ["home-3", "home-1", "home-2"],
    "purchase_price": [420_000_000, 510_000_000, 350_000_000],
    "purchase_loan_principal": [None, 270_000_000, 150_000_000],
    "purchase_loan_monthly_repayment": [None, 1_600_000, 800_000],
    "purchase_loan_rate": [None, 4.2, 3.9],
    "purchase_loan_term_months": [None, 360, 300],
})
loans = pd.DataFrame({
    "real_asset_id": ["home-1", "home-1", "home-2", "home-2", None],
    "housing_purchase_linked": [1, 1, 1, 0, 0],
    "original_principal": [250_000_000, 20_000_000, 150_000_000, 10_000_000, 8_000_000],
    "monthly_repayment": [1_400_000, 200_000, 800_000, 230_000, 160_000],
})
regions = pd.DataFrame([
    ("demo-001", "서울특별시 성북구", None, "서울특별시", "성북구", None),
    ("demo-002", "경기도 수원시 영통구", "경기도", "수원시", "영통구", None),
    ("demo-003", "경기도 용인시 수지구", "경기도", "용인시", "수지구", None),
    ("demo-001", "서울특별시 성북구", None, "서울특별시", "성북구", None),
], columns=["customer_id", "full_region_name", "sido", "si", "gu", "gun"])

assert home["real_asset_id"].is_unique
assert home["customer_id"].is_unique

**구매 연결 대출 집계와 결측 처리**

구매 시점의 자금 규모를 비교하기 위해 최초 대출 원금과 월 상환액을 집계합니다. 구매에 연결된 대출이 없는 행은 원금·상환액을 0으로 채우고, 적용 대상이 없는 금리·만기는 결측으로 유지합니다.

In [2]:
amounts = ["purchase_loan_principal", "purchase_loan_monthly_repayment"]
linked = loans.loc[loans["housing_purchase_linked"].eq(1)]
assert linked["real_asset_id"].notna().all()
assert linked["real_asset_id"].isin(home["real_asset_id"]).all()
rollup = linked.groupby("real_asset_id", as_index=False).agg(
    purchase_loan_principal=("original_principal", "sum"),
    purchase_loan_monthly_repayment=("monthly_repayment", "sum"),
)

prepared = home.copy()
no_purchase_loan = ~prepared["real_asset_id"].isin(rollup["real_asset_id"])
prepared.loc[no_purchase_loan, amounts] = prepared.loc[no_purchase_loan, amounts].fillna(0)
assert prepared.loc[no_purchase_loan, ["purchase_loan_rate", "purchase_loan_term_months"]].isna().all().all()

comparison = prepared[["real_asset_id", *amounts]].merge(
    rollup, on="real_asset_id", how="left", validate="one_to_one",
    suffixes=("_purchase", "_detail"), indicator=True,
)
for col in amounts:
    expected = comparison[f"{col}_detail"].where(comparison["_merge"].eq("both"), 0)
    assert comparison[f"{col}_purchase"].eq(expected).all(), f"집계 불일치: {col}"
print("구매 연결 대출 원금·월 상환액 집계 일치")

구매 연결 대출 원금·월 상환액 집계 일치


**지역 정보 병합과 결과 검증**

지역 정보에서 완전히 같은 중복 행을 제거한 뒤 고객 ID가 유일한지 확인합니다. 지역 정보를 왼쪽 병합한 후 행 수와 원본 열의 값·순서, 매칭 누락 여부를 검사합니다.

In [3]:
region_columns = ["full_region_name", "sido", "si", "gu", "gun"]
region_map = regions[["customer_id", *region_columns]].drop_duplicates()
assert region_map["customer_id"].is_unique, "동일 고객의 지역 정보가 서로 다릅니다."

merged = prepared.merge(
    region_map, on="customer_id", how="left",
    sort=False, validate="one_to_one", indicator=True,
)
unmatched = int(merged["_merge"].ne("both").sum())
assert len(merged) == len(prepared)
assert unmatched == 0
assert_frame_equal(merged[prepared.columns], prepared.reset_index(drop=True))

result = merged.drop(columns="_merge")
print(f"예제 검증: {len(result)}행 유지, 지역 정보 {len(region_columns)}개 추가, 매칭 누락 {unmatched}건")

예제 검증: 3행 유지, 지역 정보 5개 추가, 매칭 누락 0건


**실제 프로젝트 검증 결과**

기존 분석 기록에서는 주택 구매 데이터 **42,225행**에 지역 정보 **5개 컬럼**을 추가했습니다. 병합 전후 원본 값과 행 순서가 유지됐고, 고객 ID 중복과 지역 매칭 실패는 각각 **0건**이었습니다. 이 수치는 위 가상 예제에서 산출한 값과 별개입니다.

원본·전달 자료의 출처와 대출 계산 기준은 [데이터 결합 검증 보고서](data-lineage.md)를 참고하세요.